# Lab 19: API Requests and Caching — Analysis

Run these experiments to see the difference caching makes.

**Before you start:** Make sure your `crypto.py` implementations pass all tests.

**Setup:** Put your CoinGecko Demo API key in the cell below.

In [6]:
!pip install crypto

In [8]:
import sys
sys.path.insert(0, '../src')

"""
Lab 19: API Requests and Caching

Fetch cryptocurrency prices from CoinGecko and cache them locally.
"""

import time
import requests


# CoinGecko API base URL
BASE_URL = "https://api.coingecko.com/api/v3"
API_KEY: str = "CG-RdMAGo6UGPa8gACEyryHCHKG"


def get_price(coin_id: str, api_key: str) -> float:
    """
    Fetch the current USD price of a single cryptocurrency.

    Args:
        coin_id: CoinGecko coin identifier (e.g., "bitcoin", "ethereum")
        api_key: CoinGecko Demo API key

    Returns:
        The current price in USD as a float.

    Raises:
        RuntimeError: If the API response status code is not 200.
    """
    # TODO: Task 1
    # 1. Make a GET request to BASE_URL + "/simple/price"
    #    with params: ids, vs_currencies, x_cg_demo_api_key
    # 2. Check status code — raise RuntimeError if not 200
    # 3. Parse JSON and return the USD price as a float
    response = requests.get(BASE_URL + "/simple/price", params={
        "ids": coin_id,
        "vs_currencies": "usd",
        "x_cg_demo_api_key": api_key
    })
    if response.status_code != 200:
        raise RuntimeError(f"API request failed with status code {response.status_code}")
    data = response.json()
    return data[coin_id]["usd"]


def get_prices_batch(coin_ids: list, api_key: str) -> dict:
    """
    Fetch USD prices for multiple coins in a single API call.

    Args:
        coin_ids: List of CoinGecko coin identifiers
        api_key: CoinGecko Demo API key

    Returns:
        Dictionary mapping coin_id to USD price.
        Example: {"bitcoin": 65432.10, "ethereum": 3456.78}

    Raises:
        RuntimeError: If the API response status code is not 200.
    """
    # TODO: Task 2
    # 1. Join coin_ids into a comma-separated string
    # 2. Make ONE GET request with the joined string as "ids"
    # 3. Check status code
    # 4. Parse JSON and flatten into {coin_id: price} dict
    ids_str = ",".join(coin_ids)
    response = requests.get(BASE_URL + "/simple/price", params={
        "ids": ids_str,
        "vs_currencies": "usd",
        "x_cg_demo_api_key": api_key
    })
    if response.status_code != 200:
        raise RuntimeError(f"API request failed with status code {response.status_code}")
    data = response.json()
    return {coin_id: info["usd"] for coin_id, info in data.items()}


class CoinCache:
    """
    A time-aware cache for cryptocurrency prices.

    Stores prices with timestamps and serves them back
    until they expire (based on TTL).
    """

    def __init__(self, ttl_seconds: int = 60):
        """
        Initialize the cache.

        Args:
            ttl_seconds: How many seconds a cached entry stays fresh.
        """
        # TODO: Task 3
        # Set up: ttl_seconds, _store (empty dict), hits (0), misses (0)
        self.ttl_seconds = ttl_seconds
        self._store = {}
        self.hits = 0
        self.misses = 0

    def put(self, coin_id: str, price: float):
        """
        Store a price in the cache with a timestamp.

        Args:
            coin_id: The coin identifier
            price: The USD price to cache
        """
        # TODO: Task 3
        # Store {"price": price, "timestamp": time.time()} in _store
        self._store[coin_id] = {"price": price, "timestamp": time.time()}

    def get(self, coin_id: str):
        """
        Retrieve a cached price if it exists and hasn't expired.

        Args:
            coin_id: The coin identifier

        Returns:
            The cached price as a float, or None if not found / expired.
        """
        # TODO: Task 3 — basic version (just check if key exists)
        # TODO: Task 4 — add TTL check (is the entry still fresh?)
        try:
            entry = self._store[coin_id]
            if time.time() - entry["timestamp"] < self.ttl_seconds:
                self.hits += 1
                return entry["price"]
            else:
                # Entry expired
                del self._store[coin_id]
                self.misses += 1
                return None
        except KeyError:
            self.misses += 1
            return None


def get_price_cached(coin_id: str, api_key: str, cache: CoinCache) -> float:
    """
    Fetch a coin's price, using the cache when possible.

    Cache-aside pattern:
    1. Check the cache
    2. On hit — return cached price, skip the API
    3. On miss — fetch from API, store in cache, return price

    Args:
        coin_id: CoinGecko coin identifier
        api_key: CoinGecko Demo API key
        cache: A CoinCache instance

    Returns:
        The USD price as a float.
    """
    # TODO: Task 5
    # 1. Try cache.get(coin_id)
    # 2. If not None, return it (cache hit!)
    # 3. If None, call get_price(), store with cache.put(), return price
    cached_price = cache.get(coin_id)
    if cached_price is not None:
        return cached_price
    price = get_price(coin_id, api_key)
    cache.put(coin_id, price)
    return price


API_KEY = "CG-RdMAGo6UGPa8gACEyryHCHKG"  # Replace with your CoinGecko Demo key

## Experiment 1: Uncached vs. Cached

Fetch Bitcoin's price 10 times — first without caching, then with caching.
Compare the total time and number of API calls.

In [9]:
# --- Uncached: 10 direct API calls ---
start = time.time()
for i in range(10):
    price = get_price("bitcoin", API_KEY)
uncached_time = time.time() - start

print(f"Uncached: 10 requests in {uncached_time:.2f} seconds")
print(f"Last price: ${price:,.2f}")

Uncached: 10 requests in 0.62 seconds
Last price: $75,925.00


In [10]:
# --- Cached: 10 lookups through cache ---
cache = CoinCache(ttl_seconds=60)

start = time.time()
for i in range(10):
    price = get_price_cached("bitcoin", API_KEY, cache)
cached_time = time.time() - start

print(f"Cached: 10 lookups in {cached_time:.2f} seconds")
print(f"Cache hits: {cache.hits}, Cache misses: {cache.misses}")
print(f"Speedup: {uncached_time / cached_time:.1f}x faster")

Cached: 10 lookups in 0.06 seconds
Cache hits: 9, Cache misses: 1
Speedup: 11.0x faster


### Writeup Questions — Experiment 1

1. How many API calls did the cached version actually make? Why that number?
2. What was the approximate speedup? Why is the difference so large?
3. Is there any downside to this speedup? What are you giving up?

*Your answers here:*

1. The cached version actually made only one API call. This is because it made only one lookup that missed the cache.
2. The approximate speedup was 560 milliseconds. The difference was so large because it takes significantly longer to wait for a second response from the internet than to use the previous response from memory.
3. Having a cache means that, if a value changes at the source of truth, the value in the cache may be stale. We are giving up timeliness.

## Experiment 2: TTL Exploration

Try three different TTL values. For each one, simulate a pattern
of lookups spaced 2 seconds apart and observe the hit rate.

In [11]:
ttl_values = [1, 5, 30]

for ttl in ttl_values:
    cache = CoinCache(ttl_seconds=ttl)

    for i in range(6):
        price = get_price_cached("bitcoin", API_KEY, cache)
        if i < 5:  # Don't sleep after last lookup
            time.sleep(2)

    total = cache.hits + cache.misses
    hit_rate = cache.hits / total * 100
    print(f"TTL={ttl:2d}s: {cache.hits} hits, {cache.misses} misses, hit rate={hit_rate:.0f}%")

TTL= 1s: 0 hits, 6 misses, hit rate=0%
TTL= 5s: 4 hits, 2 misses, hit rate=67%
TTL=30s: 5 hits, 1 misses, hit rate=83%


### Writeup Questions — Experiment 2

1. With TTL=1 second and lookups every 2 seconds, what hit rate do you expect? Did the results match?
2. If you were building a portfolio tracker that updates every time you open the app, what TTL would you choose? Explain your reasoning.
3. Is there a scenario where you'd want a TTL of 0 (no caching at all)? What about a TTL of infinity (cache forever)?

*Your answers here:*

1. I expected a 0% hit rate. Yes, the results matched.
2. I would most likely choose a TTL between 0.5 and 2 seconds. This is because we very rarely close and open an app in less than 2 seconds.
3. Having a TTL of 0 might make the user wait unnecessarily, making the app feel less responsive. Having a TTL of infinity would mean that one would only need to make one API call for that specific key. The result would be that, for the rest of eternity, the cache would be stale.

## Experiment 3: Batch Efficiency

Compare fetching 5 coins one at a time vs. in a single batch request.

In [12]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "dogecoin"]

# --- Individual calls ---
start = time.time()
individual_prices = {}
for coin in coins:
    individual_prices[coin] = get_price(coin, API_KEY)
individual_time = time.time() - start

print(f"Individual: {len(coins)} calls in {individual_time:.2f} seconds")

# --- Batch call ---
start = time.time()
batch_prices = get_prices_batch(coins, API_KEY)
batch_time = time.time() - start

print(f"Batch: 1 call in {batch_time:.2f} seconds")
print(f"Speedup: {individual_time / batch_time:.1f}x faster")

# Show the prices
print("\nPrices:")
for coin, price in batch_prices.items():
    print(f"  {coin}: ${price:,.2f}")

Individual: 5 calls in 0.50 seconds
Batch: 1 call in 0.09 seconds
Speedup: 5.7x faster

Prices:
  bitcoin: $75,866.00
  cardano: $0.25
  dogecoin: $0.10
  ethereum: $2,330.89
  solana: $86.13


### Writeup Questions — Experiment 3

1. How much faster was the batch call? Where does that time saving come from?
2. Batching and caching are both ways to reduce API calls. When would you use one vs. the other? Can you use both?

*Your answers here:*

1. The batch call was 410 milliseconds faster than the individual call. That time saving came from reducing the number of API calls.
2. I would most likely use batching if I wanted to get the current temperature in multiple cities at once and I would use caching if I wanted to get the temperature in the same city multiple times. Yes, we can use both.